## Importing Libraries

In [1]:
import os
import shutil
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from multiprocessing import cpu_count
import pandas as pd
from tqdm import tqdm

## Path Variables

In [2]:
# ---------------- CONFIG ----------------
excel_path = r"C:\Users\chethan.m\Desktop\90+ ODR\90+ Pan India ODR Data.xlsx" # <-- update
output_folder = r"C:\Users\chethan.m\Desktop\90+ ODR\ODR"
temp_output_folder = r"C:\Users\chethan.m\Desktop\90+ ODR\TEMP"
signature_path = r"C:\Users\chethan.m\Desktop\90+ ODR\Sign Of Advocate.png"
background_path = r"C:\Users\chethan.m\Desktop\90+ ODR\Advocate BG.png"

## Number of Workers (Change Batch Size Only)

In [3]:
MAX_WORKERS = max(1, cpu_count() - 1)
BATCH_SIZE = 6000

## Generation (Never Change)

In [4]:
MAX_WORKERS = max(1, cpu_count() - 1)
BATCH_SIZE = 2000

os.makedirs(output_folder, exist_ok=True)
os.makedirs(temp_output_folder, exist_ok=True)
logging.basicConfig(filename='pdf_generation.log', level=logging.INFO,
                    format='%(asctime)s %(levelname)s %(message)s')
failure_log_path = "failed_rows.csv"

# ---------------- TEMPLATE ----------------
LETTER_BODY = """«Sl_No»<br/>
<b>«Name_of_the_Borrower»</b><br/>
«Mobile_Number»<br/>

Madam/Sir,

<b> REF: INVITATION TO CONCILIATION AS PER PROVISIONS OF SECTION 62 OF THE ARBITRATION AND CONCILIATION ACT, 1996 / YOUR LOAN ACCOUNT No. «LAN». </b>

1. Representing our clients, CapFloat Financial Services Private Limited (Formerly Zen Lefin Private Limited, earlier brand name ‘Capital Float’ and now operating under the brand name “axio”), having its registered office at, Gokaldas Platinum, New No.3, Old No.211, Bellary Road, Upper Palace Orchards, Sadashivanagar, Bengaluru - 560080, (herein after referred to as the Company) we address you as under: 

2. The Company represents to us that you had applied for loan facility from our client and you were disbursed with the same through the aforementioned captioned Loan Account No.(s) (hereinafter referred to as “the said Loan”). Thereafter, you had also utilized the said Loan for your end use / benefits inter alia agreeing to abide by the terms and conditions of the Loan Agreement.

3. The company further states that after availing the loan facility, you have failed to keep up your promise as per the agreed terms and conditions of the loan agreement and have been evading the requisite payments due on the said loan Account(s) in spite of repeated requests, reminders and personal visits by the representatives of the Company. You are requested to note that you have neither complied with the terms & conditions of the Loan Agreement nor you have come up with any proposal for repaying the outstanding on the said Loan Account. As on «As_on» you are due and liable to pay to our client an Outstanding Amount of Rs. «Closeout_Amount»/- inclusive of interest and other applicable charges.

4. Since the matter has now been entrusted to us, we hereby request you to appear for a Conciliation Proceeding before the Conciliator. Therefore, you are instructed to appear online and meet «Conciliator» Conciliator, on <b>«Date_of_ODR»</b>, between 10AM to 5PM. The web-link to join the online conciliation proceedings will be shared separately to your contact number.

5. During the said Conciliation Proceeding, you may express your difficulties in repayment of Outstanding Dues for reaching an amicable settlement/resolution, which would be mutually acceptable with your co-operation. The concerned Officials of our client will also be present for the same.

6. Please note that in the event you fail to respond to this Conciliation Notice, on the expiry of Seven (7) working days from the date of receipt of this notice, the Company will initiate Civil and Criminal proceedings against you, which would be entirely at your risk, cost and consequences thereof.

7. This invitation is without prejudice to our client’s rights and contentions that may be available under law.

8. Please ignore this communication if the matter is already resolved / closed / if amount demanded above is already paid.

Note: Please contact for any further clarification  
«FTE_Number»
"""

# ---------------- PDF BACKGROUND ----------------
def add_background(canvas, doc):
    if background_path and os.path.exists(background_path):
        from reportlab.lib.pagesizes import A4
        page_width, page_height = A4
        canvas.saveState()
        canvas.drawImage(background_path, 0, 0, width=page_width, height=page_height, preserveAspectRatio=True, mask='auto')
        canvas.restoreState()

# ---------------- WORKER FUNCTION ----------------
def generate_pdf_from_row(row_dict, temp_output_folder, signature_path):
    try:
        from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image
        from reportlab.lib.pagesizes import A4
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
        from reportlab.lib.enums import TA_JUSTIFY, TA_LEFT, TA_RIGHT

        styles = getSampleStyleSheet()
        styles.add(ParagraphStyle(name='Justify', alignment=TA_JUSTIFY,fontSize=8,leading=14))
        styles.add(ParagraphStyle(name='Left', alignment=TA_LEFT,fontSize=8, leading=14))
        styles.add(ParagraphStyle(name='Right', alignment=TA_RIGHT, fontSize=8,leading=14))

        # Map Excel columns properly
        sl_no = str(row_dict.get("Sl. No", "")).strip()
        lan = str(row_dict.get("LAN", "")).strip()
        name = str(row_dict.get("Name of the Borrower", "")).strip()
        mobile = str(row_dict.get("Mobile Number", "")).strip()
        as_on = str(row_dict.get("As on", "")).strip()
        closeout_amount = str(row_dict.get("Closeout Amount", "")).strip()
        conciliator = str(row_dict.get("Conciliator", "")).strip()
        date_of_odr = str(row_dict.get("Date of ODR", "")).strip()
        fte_number = str(row_dict.get("FTE Number", "")).strip()

        # Replace placeholders
        letter_text = LETTER_BODY.replace("«Sl_No»", sl_no)\
                                 .replace("«LAN»", lan)\
                                 .replace("«Name_of_the_Borrower»", name)\
                                 .replace("«Mobile_Number»", mobile)\
                                 .replace("«As_on»", as_on)\
                                 .replace("«Closeout_Amount»", closeout_amount)\
                                 .replace("«Conciliator»", conciliator)\
                                 .replace("«Date_of_ODR»", date_of_odr)\
                                 .replace("«FTE_Number»", fte_number)

        filename = f"{lan}.pdf" if lan else f"row_{sl_no}.pdf"
        tmp_path = os.path.join(temp_output_folder, filename)

        # Wider paragraph by reducing left/right margins
        doc = SimpleDocTemplate(tmp_path, pagesize=A4,
                                rightMargin=36, leftMargin=36,   # reduced from 72
                                topMargin=72+19,
                                bottomMargin=72)

        story = []
        for para in letter_text.split("\n\n"):
            if para.strip():
                story.append(Paragraph(para.strip(), styles['Justify']))
                story.append(Spacer(1, 12))

        # Signature on RIGHT
        if signature_path and os.path.exists(signature_path):
            img = Image(signature_path, width=120, height=50, hAlign='RIGHT')
            story.append(img)
            story.append(Spacer(1, 12))

        story.append(Paragraph("Authorised Signatory", styles['Right']))

        doc.build(story, onFirstPage=add_background, onLaterPages=add_background)

        return True, lan, tmp_path

    except Exception as e:
        return False, str(row_dict.get("LAN", row_dict.get("Sl. No", "unknown"))), str(e)

# ---------------- BATCH RUNNER ----------------
def run_in_batches(df, temp_output_folder, output_folder, signature_path,
                   max_workers=MAX_WORKERS, batch_size=BATCH_SIZE):
    rows = df.to_dict(orient="records")
    failures = []

    for batch_start in range(0, len(rows), batch_size):
        batch = rows[batch_start: batch_start + batch_size]

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(generate_pdf_from_row, row, temp_output_folder, signature_path): row for row in batch}

            for fut in tqdm(as_completed(futures), total=len(futures),
                            desc=f"Batch {batch_start+1}-{min(batch_start+batch_size,len(rows))}"):
                row = futures[fut]
                try:
                    success, lan_or_id, result = fut.result()
                    if success:
                        final_path = os.path.join(output_folder, os.path.basename(result))
                        shutil.move(result, final_path)
                    else:
                        logging.error(f"Failed {lan_or_id}: {result}")
                        failures.append({**row, "error": result})
                except Exception as e:
                    logging.exception("Unhandled exception in future")
                    failures.append({**row, "error": str(e)})

    if failures:
        pd.DataFrame(failures).to_csv(failure_log_path, index=False)
        print(f"Completed with {len(failures)} failures. See {failure_log_path} and pdf_generation.log")
    else:
        print("Completed successfully with zero failures.")

# ---------------- MAIN ----------------
if __name__ == "__main__":
    df = pd.read_excel(excel_path)
    df.columns = [col.strip() for col in df.columns]  # Keep original column names with spaces

    # Convert 'Date of ODR' to DD-MMM-YYYY format
    if 'As on' in df.columns:
        df['As on'] = pd.to_datetime(df['As on'], errors='coerce').dt.strftime("%d-%b-%Y").fillna('')

    os.makedirs(temp_output_folder, exist_ok=True)
    os.makedirs(output_folder, exist_ok=True)

    run_in_batches(df, temp_output_folder, output_folder, signature_path,
                   max_workers=MAX_WORKERS, batch_size=BATCH_SIZE)


Batch 12001-12982: 100%|█████████████████████████████████████████████████████████████| 982/982 [02:30<00:00,  6.53it/s]

Completed successfully with zero failures.
